In [9]:
import sys, os
sys.path.append(os.path.join('/home/module'))
import pgd_imp_old as imp
import pandas as pd

import math
import pandas as pd
from google.cloud import bigquery
from google.oauth2 import service_account

# setup BigQuery client sekali
key_path = '/home/jupyterhub/SA/jupyter-pgd-prod-data-analytics-0fa0e01ee70a.json'
credentials = service_account.Credentials.from_service_account_file(
    key_path, scopes=["https://www.googleapis.com/auth/cloud-platform"],
)
bq_client = bigquery.Client(credentials=credentials, project=credentials.project_id)

In [10]:
destination_table = "pgd-dev-data-analytics.datamart_audit.dm_tbl_sidik_det_suspicious_emp"
summary_table = "pgd-dev-data-analytics.datamart_audit.dm_tbl_sidik_sum_suspicious_emp"
case_table = "pgd-dev-data-analytics.datamart_audit.dm_tbl_sidik_case_suspicious_emp"
emp_detail = "pgd-dev-data-analytics.datamart_audit.dm_tbl_emp_detail"
up_kontrak_det = "pgd-dev-data-analytics.datamart_audit.dm_grafik_sidik_up_kontrak_emp"
bj_det = "pgd-dev-data-analytics.datamart_audit.dm_grafik_sidik_bj_emp"

In [11]:
# current_date = '2026-04-08'

In [12]:
def import_detail_symptom(df,dest_table):

    if df.empty:
        print(f"DataFrame is empty. Skipping upload to {dest_table}\n")
        return  # Exit the function early
    df['symptoms'] = df['symptoms'].astype(str)
    df['alert_source'] = df['alert_source'].astype(str)
    df['nik_pendek'] = df['nik_pendek'].astype(str)
    
    df['sub_branch_cd'] = df['sub_branch_cd'].astype(str)
    df['tingkat_risiko'] = df['tingkat_risiko'].astype(str)
    df['detection_date'] = pd.to_datetime(df['detection_date'], utc=True).dt.as_unit('us')

    
    # mapping pandas dtype ke BQ type
    dtype_to_bq = {
        'object': 'STRING',
        'int64': 'INT64',
        'Int64': 'INT64',
        'int32' : 'INT64',
        'float64': 'FLOAT64',
        'float32': 'FLOAT64',
        'datetime64[ns]': 'TIMESTAMP',
        'datetime64[us, UTC]':'TIMESTAMP',
        'dbdate':'TIMESTAMP'
    }

    # bangun schema dynamically
    schema = []
    for col, dtype in df.dtypes.items():
        dt = str(dtype)
        bq_type = dtype_to_bq.get(dt)
        if not bq_type:
            raise ValueError(f"Unknown dtype '{dt}' for column '{col}'")
        schema.append(bigquery.SchemaField(col, bq_type))

    # config load job
    job_config = bigquery.LoadJobConfig(
        schema=schema,
        write_disposition="WRITE_APPEND"        
        #write_disposition="WRITE_TRUNCATE" if i == 0 else "WRITE_APPEND"
    )

    # load ke BQ
    job = bq_client.load_table_from_dataframe(df, dest_table, job_config=job_config)
    job.result()
    print(f"loaded {job.output_rows} rows\n to", dest_table) 
    

In [13]:
def import_sum_symptom(df,dest_table):

    if df.empty:
        print(f"DataFrame is empty. Skipping upload to {dest_table}\n")
        return  # Exit the function early
    
    # mapping pandas dtype ke BQ type
    dtype_to_bq = {
        'object': 'STRING',
        'int64': 'INT64',
        'Int64': 'INT64',
        'int32' : 'INT64',
        'float64': 'FLOAT64',
        'float32': 'FLOAT64',
        'datetime64[ns]': 'TIMESTAMP',
        'datetime64[us, UTC]':'TIMESTAMP',
        'datetime64[us]':'TIMESTAMP',
        'dbdate':'TIMESTAMP'
    }

    # bangun schema dynamically
    schema = []
    for col, dtype in df.dtypes.items():
        dt = str(dtype)
        bq_type = dtype_to_bq.get(dt)
        if not bq_type:
            raise ValueError(f"Unknown dtype '{dt}' for column '{col}'")
        schema.append(bigquery.SchemaField(col, bq_type))

    # config load job
    job_config = bigquery.LoadJobConfig(
        schema=schema,
        write_disposition="WRITE_APPEND"        
        #write_disposition="WRITE_TRUNCATE" if i == 0 else "WRITE_APPEND"
    )

    # load ke BQ
    job = bq_client.load_table_from_dataframe(df, dest_table, job_config=job_config)
    job.result()
    print(f"loaded {job.output_rows} rows\n to", dest_table) 
    

In [ ]:
# create_table = f"""
# CREATE TABLE IF NOT EXISTS {destination_table} 
# (
#     symptoms STRING,
#     alert_source STRING,
#     nik_pendek STRING,
#     sub_branch_cd STRING,
#     tingkat_risiko STRING,
#     detection_date TIMESTAMP
# );

# """
#symptoms	alert_source	nik_pendek	sub_branch_cd	tingkat_risiko	detection_date
    
#drop_table = f"""drop table {destination_table}"""

#bq_client.query(drop_table)
# bq_client.query(create_table).result()

In [ ]:
#Summary Table - list symptom yang terflag per nik_pendek -- per hari

# create_table = f"""
# CREATE TABLE IF NOT EXISTS {summary_table} (
#   symptoms STRING,
#     alert_source STRING,
#     nik_pendek STRING,
#     sub_branch_cd STRING,
#     tingkat_risiko STRING,
#     detection_date TIMESTAMP,
#     jumlah_symptoms INT64
# );

# """
# bq_client.query(create_table)
# bq_client.query(create_table).result()

In [ ]:
# Case_ID

# create_table = f"""
# CREATE TABLE IF NOT EXISTS {case_table} (
#   case_id STRING,
#   detection_date DATE,
#   nik_pendek STRING,
#   symptoms STRING,
#   alert_source STRING, 
#   status STRING
# );

# """
# bq_client.query(create_table).result()

In [3]:
# create_table = f"""
# CREATE TABLE IF NOT EXISTS {bj_det} (
#     terkahir_proses STRING,
#     terakhir_date TIMESTAMP,
#     tot_bj INT64

# );
# """

In [ ]:
# create_destable = f"""
# create table {destination_table}
# """

# sql.execute_ddl(create_destable)

### Tabel Employee Detail

In [10]:
kueri = f"""

SELECT 
    tk.create_by,
    k.update_by,
    k.approval_by,
    k.analyzed_by,
    tk.tgl_transaksi,
    --k.tgl_fpk,
    --k.tgl_kontrak,
    --k.tgl_lunas,
    case 
        when h.description like '%CAIR%' then k.create_date
        else tk.tgl_transaksi
    end as create_date,
    case 
        when h.description like '%CAIR%' then k.update_date
        else NULL
    end as update_date,
    k.cif,
    tk.no_kontrak,
    h.description,
    k.up,
    vt.sub_branch_nm,
    vt.branch_nm

FROM datalake.pgd_tbl_transaksi_kredit tk
    left join datalake.pgd_tbl_transaction_handler h 
        ON tk.kode_transaksi = h.kode_transaksi
    left join datalake.pgd_tbl_kredit k
        on tk.no_kontrak = k.no_kontrak
    left join datalake.pegadaian_vt_lookup_branch vt
        on tk.branch_code = vt.sub_branch_cd
where
    tk.tgl_transaksi >= date_sub(current_date(), interval 30 day)
    -- and
    --     (
    --     upper(k.create_by) = 'PKS01675'
    --     or upper(k.update_by) = 'PKS01675'
    --     or upper(k.approval_by) = 'PKS01675'
    --     or upper(k.analyzed_by) = 'PKS01675'
    -- )
order by k.create_date desc;
"""

In [11]:
df_emp_det = bq_client.query(kueri).to_dataframe()

In [12]:
df_emp_det.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 136527 entries, 0 to 136526
Data columns (total 13 columns):
 #   Column         Non-Null Count   Dtype         
---  ------         --------------   -----         
 0   create_by      136527 non-null  object        
 1   update_by      23132 non-null   object        
 2   approval_by    9832 non-null    object        
 3   analyzed_by    3090 non-null    object        
 4   tgl_transaksi  136527 non-null  datetime64[us]
 5   create_date    136275 non-null  datetime64[us]
 6   update_date    0 non-null       datetime64[us]
 7   cif            23132 non-null   object        
 8   no_kontrak     136527 non-null  object        
 9   description    136527 non-null  object        
 10  up             23132 non-null   object        
 11  sub_branch_nm  0 non-null       object        
 12  branch_nm      0 non-null       object        
dtypes: datetime64[us](3), object(10)
memory usage: 13.5+ MB


In [13]:
df_emp_det.head(1)

,create_by,update_by,approval_by,analyzed_by,tgl_transaksi,create_date,update_date,cif,no_kontrak,description,up,sub_branch_nm,branch_nm
0,P94469,None,None,None,2026-03-24,NaT,NaT,None,1116926010019888,PENCAIRAN GADAI NON TUNAI,None,None,None


In [ ]:
df_emp_det['up'] = pd.to_numeric(df_emp_det['up'], errors='coerce')
df_emp_det['create_by'] = df_emp_det['create_by'].astype(str)
df_emp_det['update_by'] = df_emp_det['update_by'].astype(str)
df_emp_det['approval_by'] = df_emp_det['approval_by'].astype(str)
df_emp_det['analyzed_by'] = df_emp_det['analyzed_by'].astype(str)
df_emp_det['cif'] = df_emp_det['cif'].astype(str)
df_emp_det['no_kontrak'] = df_emp_det['no_kontrak'].astype(str)
df_emp_det['description'] = df_emp_det['description'].astype(str)
df_emp_det['sub_branch_nm'] = df_emp_det['sub_branch_nm'].astype(str)
df_emp_det['branch_nm'] = df_emp_det['branch_nm'].astype(str)


In [ ]:
import_sum_symptom(df_emp_det,emp_detail)

### Tabel untuk grafik historis up, jumlah kontrak 

In [ ]:
up_kontrak = f"""

with up_kontrak as (
SELECT
    coalesce(k.update_by, tk.create_by) terkahir_proses,
    coalesce(k.update_date, tk.tgl_transaksi) terakhir_date,
    tk.no_kontrak,
    k.up
FROM pgd-prod-data-analytics.datalake.pgd_tbl_transaksi_kredit tk
    left join datalake.pgd_tbl_transaction_handler h 
        ON tk.kode_transaksi = h.kode_transaksi
    left join datalake.pgd_tbl_kredit k
        on tk.no_kontrak = k.no_kontrak
    
where

    tk.tgl_transaksi >= date_sub(current_date(), interval 30 day)

order by terakhir_date desc
)
  select
  terkahir_proses,
  date(terakhir_date) terakhir_date, 
  count(no_kontrak) jumlah_kontrak,
  sum(up) total_up,
  from up_kontrak
  group by terkahir_proses, terakhir_date
  order by terkahir_proses,  terakhir_date desc

"""

In [ ]:
df_up_kontrak = bq_client.query(up_kontrak).to_dataframe()

In [ ]:
df_up_kontrak.info()

In [ ]:
df_up_kontrak['total_up'] = pd.to_numeric(df_up_kontrak['total_up'], errors='coerce')
df_up_kontrak['terkahir_proses'] = df_up_kontrak['terkahir_proses'].astype(str)
df_up_kontrak['terakhir_date'] = df_up_kontrak['terakhir_date']

In [ ]:
import_sum_symptom(df_up_kontrak,up_kontrak_det)

### Tabel untuk grafik historis jumlah bj

In [5]:
bj = """
with data_jam as (
  select
    account_no,
    jumlah,
    coalesce(update_by, create_by) terkahir_proses,
    coalesce(update_date, create_date) terakhir_date
  from pgd-prod-data-analytics.datalake.pgd_tbl_det_jam_kt
  where 
    greatest(
      date(create_date), 
      date(update_date), 
      date(tgl_taksir)
      ) >= date_sub(current_datetime('Asia/Jakarta'), interval 30 day)

  union distinct

  select
    account_no,
    jumlah,
    coalesce(update_by, create_by) terkahir_proses,
    coalesce(update_date, create_date) terakhir_date
  from pgd-prod-data-analytics.datalake.pgd_tbl_det_jam_el
  where 
    greatest(
      date(create_date), 
      date(update_date), 
      date(tgl_taksir)
      ) >= date_sub(current_datetime('Asia/Jakarta'), interval 30 day)

  union distinct

  select
    account_no,
    jumlah,
    coalesce(update_by, create_by) terkahir_proses,
    coalesce(update_date, create_date) terakhir_date
  from pgd-prod-data-analytics.datalake.pgd_tbl_det_jam_gd
  where 
    greatest(
      date(create_date), 
      date(update_date), 
      date(tgl_taksir)
      ) >= date_sub(current_datetime('Asia/Jakarta'), interval 30 day)
  

  union distinct

  select
    account_no,
    jumlah,
    coalesce(update_by, create_by) terkahir_proses,
    coalesce(update_date, create_date) terakhir_date
  from pgd-prod-data-analytics.datalake.pgd_tbl_det_jam_ps
   where 
     greatest(
       date(create_date), 
       date(update_date)
       ) >= date_sub(current_datetime('Asia/Jakarta'), interval 30 day)

  union distinct

  select
    account_no,
    jumlah,
    coalesce(update_by, create_by) terkahir_proses,
    coalesce(update_date, create_date) terakhir_date
  from pgd-prod-data-analytics.datalake.pgd_tbl_det_jam_sb
   where 
     greatest(
       date(create_date), 
       date(update_date)
       ) >= date_sub(current_datetime('Asia/Jakarta'), interval 30 day)
)

select 
  terkahir_proses,
  date(terakhir_date) terakhir_date,
  sum(jumlah) tot_bj
from data_jam
group by terkahir_proses, terakhir_date
order by terkahir_proses,  terakhir_date desc

"""

In [6]:
df_bj = bq_client.query(bj).to_dataframe()

In [7]:
df_bj.head(1)

,terkahir_proses,terakhir_date,tot_bj
0,ERA00333,2026-03-16,1.00000000000000000000000000000000000000


In [8]:
df_bj.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 73983 entries, 0 to 73982
Data columns (total 3 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   terkahir_proses  73983 non-null  object
 1   terakhir_date    73983 non-null  dbdate
 2   tot_bj           73970 non-null  object
dtypes: dbdate(1), object(2)
memory usage: 1.7+ MB


In [ ]:
df_bj['tot_bj'] = pd.to_numeric(df_bj['tot_bj'], errors='coerce')
df_bj['terkahir_proses'] = df_bj['terkahir_proses'].astype(str)
df_bj['terakhir_date'] = pd.to_datetime(df_bj['terakhir_date'], errors='coerce')

In [ ]:
import_sum_symptom(df_bj,bj_det)

### Symptom E-ML: 

In [14]:
# Jumlah Rekening/Kontrak Kredit dalam Seminggu 
s_1b = f"""
--INSERT INTO {destination_table}
select 
    nik_pendek,
    branch_code as sub_branch_cd,
    --value_up,
    --jumlah_bj,
    --jumlah_kontrak,
    final_label as tingkat_risiko,
    case
    when final_label='Moderate' then 3.0
    when final_label='Moderate to High' then 4.0
    when final_label='High' then 5.0
end as skor_risiko,
    detection_date,
    'RFM Uang Pinjaman Jumlah Kontrak dan Barang Jaminan oleh Machine Learning' as symptoms,
    'Machine Learning' as alert_source 
from datamart_audit.dm_dt_sidik_ml_employee
where final_label != 'Normal' and
date(detection_date) = date_sub(current_date(), interval 0 day)
"""

df_ml = bq_client.query(s_1b).to_dataframe()
print('total rows: ', len(df_ml))
print(df_ml.head(2))


total rows:  22
                                     nik_pendek sub_branch_cd  \
0  EJHBDWq04GlHeTlAOykGLTFdM+3canLdmnn05QFnf5c=         11179   
1  EJHBDWq04GlHeTlAOykGLTFdM+3canLdmnn05QFnf5c=         11179   

     tingkat_risiko detection_date  \
0  Moderate to High     2026-02-19   
1  Moderate to High     2026-02-19   

                                            symptoms      alert_source  
0  RFM Uang Pinjaman Jumlah Kontrak dan Barang Ja...  Machine Learning  
1  RFM Uang Pinjaman Jumlah Kontrak dan Barang Ja...  Machine Learning  


In [15]:
import_detail_symptom(df_ml,destination_table)

loaded 22 rows
 to pgd-dev-data-analytics.datamart_audit.dm_tbl_sidik_det_suspicious_emp


### Symptom E1: Karyawan Melakukan Pencairan >1 Unit Kerja

In [ ]:
emp_1 = """
with trx_hari_ini as
(
SELECT
tk.no_kontrak,
tk.create_by,
tk.branch_code
FROM datalake.pgd_tbl_transaksi_kredit tk
left join datalake.pgd_tbl_transaction_handler h
ON tk.kode_transaksi = h.kode_transaksi
where
h.description like '%CAIR%'
and tk.tgl_transaksi <= date_sub(current_date(),INTERVAL 30 DAY)
and tk.tgl_transaksi >= date_sub(current_date(),INTERVAL 30 DAY)
),

emp_cif as
(
-- CIF & NIK Karyawan Internal
select
cus.cif,
emp.nik_pendek,
emp.ktp,
emp.branch_code
from datalake.hcms_emp_cur_data emp
inner join datalake.pgd_tbl_customer cus
on emp.ktp = cus.no_id
where emp.branch_code is not null

union all

-- CIF dan NIK Karyawan Outsource
select
cus.cif,
ara.employee_id as nik_pendek,
ara.no_ktp as ktp,
ara.kode_location as branch_code
from datalake.aralia_public_ms_employee_temp ara -- ini kalau udah ada tabel datalake jangan pake raw
inner join datalake.pgd_tbl_customer cus
on ara.no_ktp = cus.no_id
),

suspect as
(
select
'Karyawan Melakukan Pencairan Lebih dari 1 Unit Kerja' as symptoms,
'Rule-Based' as alert_source,
emp.nik_pendek,
emp.branch_code as sub_branch_cd,
date_sub(current_date(),INTERVAL 3 DAY) as as_dt,
case
    when count (distinct hi.branch_code)=2 then 'Moderate'
    when count (distinct hi.branch_code)>2 then 'High'
end as tingkat_risiko,
case
    when count (distinct hi.branch_code)=2 then 3.0
    when count (distinct hi.branch_code)>2 then 4.0
end as skor_risiko, 
current_date() as detection_date

from
emp_cif emp
left join trx_hari_ini hi
on emp.nik_pendek = hi.create_by

where hi.branch_code is not null
group by emp.nik_pendek, as_dt, emp.branch_code
having count (distinct hi.branch_code) > 1
)


select distinct *
from suspect
"""

In [ ]:
df_emp_1 = bq_client.query(emp_1).to_dataframe()
print('total rows: ', len(df_emp_1))
print(df_emp_1.head(2))

In [ ]:
df_emp_1 = df_emp_1.drop(columns=['as_dt'])

In [ ]:
df_emp_1.info()

In [ ]:
import_detail_symptom(df_emp_1,destination_table)

# Summary & Case-ID

In [20]:
# symptoms STRING,
#     cif_karyawan STRING,
#     nik_pendek STRING,
#     sub_branch_cd STRING,
#     detection_date TIMESTAMP,
#     value_up FLOAT64,
#     value_bj FLOAT64,
#     jumlah_kontrak FLOAT64
#symptoms	alert_source	nik_pendek	sub_branch_cd	tingkat_risiko	detection_date
    
sum_table = f"""
--insert into {summary_table}
select
base.symptoms,
base.alert_source,
base.nik_pendek,
base.sub_branch_cd,
base.tingkat_risiko,
base.detection_date,
base.jumlah_symptoms


from (
    SELECT 
    nik_pendek,
    string_agg(distinct symptoms, ', ') AS symptoms,
    string_agg(distinct alert_source, ', ') AS alert_source,
    cast(count(distinct symptoms) as integer) as jumlah_symptoms,
    detection_date,
    sub_branch_cd,
    tingkat_risiko
    FROM (
        select distinct *
        from
        {destination_table}
        ) d
    where date(detection_date) = date_sub(current_date(), interval 0 day)
    group by nik_pendek, detection_date,sub_branch_cd, tingkat_risiko) base
order by base.jumlah_symptoms desc

"""
df_sum = bq_client.query(sum_table).to_dataframe()
print('total rows: ', len(df_sum))
print(df_sum.head(2))


total rows:  11
                                            symptoms      alert_source  \
0  RFM Uang Pinjaman Jumlah Kontrak dan Barang Ja...  Machine Learning   
1  RFM Uang Pinjaman Jumlah Kontrak dan Barang Ja...  Machine Learning   

                                     nik_pendek sub_branch_cd tingkat_risiko  \
0  I4rNy6xsIGzEFMh+JVJA0CTMol43DJVpSj1neh6W7lI=         11847       Moderate   
1  1gF5U80t+EagV3Aq2sNX8GqKEl94z7+rcJDFMV33zxg=         10808       Moderate   

             detection_date  jumlah_symptoms  
0 2026-02-19 00:00:00+00:00                1  
1 2026-02-19 00:00:00+00:00                1  


In [21]:
import_sum_symptom(df_sum,summary_table)

loaded 11 rows
 to pgd-dev-data-analytics.datamart_audit.dm_tbl_sidik_sum_suspicious_emp


In [22]:
# base.symptoms,
# base.alert_source,
# base.nik_pendek,
# base.sub_branch_cd as kode_outlet,
# base.tingkat_risiko,
# base.detection_date,
# base.jumlah_symptoms
c_table = f"""

WITH latest_table_case AS (
    -- Ngambil yang masuk di case_id nik_pendek-nya apa aja buat digabungin dengan table_summary
    SELECT 
        case_id,
        detection_date,
        nik_pendek,
        symptoms,
        alert_source,
        status,
        ROW_NUMBER() OVER (PARTITION BY nik_pendek ORDER BY detection_date DESC) AS rn
    FROM {case_table}
    WHERE date(detection_date) >= date_sub(current_date(), interval 30 day)
),
ranked_table_summary AS (
    -- ngambil data dari summary
    SELECT 
        date(detection_date) detection_date,
        nik_pendek,
        jumlah_symptoms,
        symptoms,
        alert_source,
        tingkat_risiko,
        LAG(symptoms) OVER (PARTITION BY nik_pendek ORDER BY detection_date) AS prev_symptoms_sum,
        LAG(jumlah_symptoms) OVER (PARTITION BY nik_pendek ORDER BY detection_date) AS prev_jumlah_symptoms_sum,
        LAG(detection_date) OVER (PARTITION BY nik_pendek ORDER BY detection_date) AS prev_detection_date_sum
    FROM {summary_table}
    where date(detection_date) >= date_sub(current_date(), interval 1 day) 
),
combined_history AS (
    -- Gabungan dari Table Summary & CaseID -- tambahin jumlah symptom buat yang case-id
    SELECT 
        nik_pendek,
        date(detection_date) detection_date,
        symptoms,
        CASE 
            WHEN LENGTH(symptoms) - LENGTH(REPLACE(symptoms, ',', '')) + 1 > 0 
            THEN LENGTH(symptoms) - LENGTH(REPLACE(symptoms, ',', '')) + 1
            ELSE 1 
        END AS jumlah_symptoms,
        'case' AS source
    FROM {case_table}
    
    UNION ALL
    
    SELECT 
        nik_pendek,
        date(detection_date) detection_date,
        symptoms,
        jumlah_symptoms,
        'summary' AS source
    FROM {summary_table}
),
latest_per_nik_pendek AS (
    -- Ambil record terakhir per nik_pendek dari gabungan table A dan B
    SELECT 
        nik_pendek,
        date(detection_date) AS last_detection_date,
        symptoms AS last_symptoms,
        jumlah_symptoms AS last_jumlah_symptoms,
        ROW_NUMBER() OVER (PARTITION BY nik_pendek ORDER BY detection_date DESC) AS rn
    FROM combined_history
),
filtered_table_summary AS (
    -- Filter data dari table B berdasarkan rules
    SELECT 
        rb.detection_date,
        rb.nik_pendek,
        rb.jumlah_symptoms,
        rb.symptoms,
        rb.alert_source,
        rb.tingkat_risiko,
        lpc.last_detection_date,
        lpc.last_symptoms,
        lpc.last_jumlah_symptoms
    FROM ranked_table_summary rb
    LEFT JOIN latest_per_nik_pendek lpc ON rb.nik_pendek = lpc.nik_pendek AND lpc.rn = 1
    WHERE 
        -- Record pertama untuk nik_pendek yang belum ada 
        lpc.nik_pendek IS NULL
        OR
        -- Symptoms berbeda DAN tidak dalam 30 hari terakhir
        (rb.symptoms != lpc.last_symptoms AND date_diff(date(rb.detection_date), date(lpc.last_detection_date), day) > 30)
        OR
        -- Symptoms berbeda DAN jumlah symptoms bertambah
        (rb.symptoms != lpc.last_symptoms AND rb.jumlah_symptoms > lpc.last_jumlah_symptoms)
),
new_cases AS (
    -- Generate case_id baru untuk data yang perlu ditambahkan
    SELECT 
        CONCAT(nik_pendek, cast(detection_date as string), cast(jumlah_symptoms as string)) AS case_id,
        detection_date,
        nik_pendek,
        symptoms,
        null AS status,
        tingkat_risiko,
        alert_source
    FROM filtered_table_summary
)
SELECT 
    case_id,
    detection_date,
    nik_pendek,
    symptoms,
    alert_source,
    cast(status as string) status,
    cast(NULL as string) assignee,
    cast(NULL as string) keterangan,
    cast(current_date() as timestamp) update_time
FROM new_cases
WHERE case_id NOT IN (SELECT case_id FROM {case_table} 
where date(detection_date) >= date_sub(current_date(), interval 30 day)) 
ORDER BY detection_date desc;
"""

df_case = bq_client.query(c_table).to_dataframe()
print('total rows: ', len(df_case))
print(df_case.head(2))


total rows:  0
Empty DataFrame
Columns: [case_id, detection_date, nik_pendek, symptoms, alert_source, status]
Index: []


In [ ]:
import_sum_symptom(df_case,case_table)